#Ingestion


In [0]:
import requests, json, time
from datetime import datetime
 
API_KEY = "IBAUAEMLTBZUEQOH"         
symbols  = ["AAPL", "MSFT", "GOOGL", "NVDA"]
BASE     = "https://www.alphavantage.co/query"
raw_path = "/Volumes/jrvs_databricks_fundamentals/default/raw"
 
# Create catalog / schema / volume if they don't exist
spark.sql("CREATE CATALOG IF NOT EXISTS jrvs_databricks_fundamentals")
spark.sql("CREATE SCHEMA  IF NOT EXISTS jrvs_databricks_fundamentals.default")
spark.sql("CREATE VOLUME  IF NOT EXISTS jrvs_databricks_fundamentals.default.raw")
 
# Creates Folder 
for sub in ("daily", "quote", "company"):
    dbutils.fs.mkdirs(f"{raw_path}/{sub}")
 
def getData(params):
    params["apikey"] = API_KEY
    try:
        r = requests.get(BASE, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"{params.get('symbol')}: {e}") # specific symbol it crahsed at 

    time.sleep(15)      # stay under 5 requests/min
    return data
 
for symbol in symbols:
    #Track based on file_time
    file_time = datetime.now().strftime("%Y%m%d_%H%M%S")
 
    daily_data   = getData({"function": "TIME_SERIES_DAILY", "symbol": symbol, "outputsize": "compact"})
    quote_data   = getData({"function": "GLOBAL_QUOTE", "symbol": symbol})
    company_data = getData({"function": "OVERVIEW", "symbol": symbol})
 
    daily_records = []

    time_series = daily_data.get("Time Series (Daily)", {})

    for date, v in time_series.items():
        record = {
            
            "symbol": symbol,
            "trade_date": date,
            "open":  v.get("1. open"),
            "high":  v.get("2. high"),
            "low":   v.get("3. low"),
            "close": v.get("4. close"),
            "volume": v.get("5. volume"),
        }

        daily_records.append(record)
 
    q = quote_data.get("Global Quote", {})
    clean_quote = {
        "symbol": q.get("01. symbol"),
        "price": q.get("05. price"),
        "volume": q.get("06. volume"),
        "latest_trading_day": q.get("07. latest trading day"),
        "previous_close": q.get("08. previous close"),
        "price_change": q.get("09. change"),
        "change_percent": q.get("10. change percent"),
    }
 
    clean_company = {
        "symbol": company_data.get("Symbol"),
        "company_name": company_data.get("Name"),
        "sector": company_data.get("Sector"),
        "industry": company_data.get("Industry"),
        "market_cap": company_data.get("MarketCapitalization"),
        "country": company_data.get("Country"),
        "exchange": company_data.get("Exchange"),
    }
 
    dbutils.fs.put(f"{raw_path}/daily/{symbol}_{file_time}.json",
                   "\n".join(json.dumps(r) for r in daily_records), True)
    dbutils.fs.put(f"{raw_path}/quote/{symbol}_{file_time}.json",
                   json.dumps(clean_quote), True)
    dbutils.fs.put(f"{raw_path}/company/{symbol}_{file_time}.json",
                   json.dumps(clean_company), True)

Wrote 14902 bytes.
Wrote 182 bytes.
Wrote 179 bytes.
Wrote 14901 bytes.
Wrote 184 bytes.
Wrote 195 bytes.
Wrote 15000 bytes.
Wrote 185 bytes.
Wrote 212 bytes.
Wrote 14996 bytes.
Wrote 184 bytes.
Wrote 181 bytes.


##Clean up cell

In [0]:

# for sub in ("daily", "quote", "company"):
#     dbutils.fs.rm(f"{raw_path}/{sub}", True)
#     dbutils.fs.mkdirs(f"{raw_path}/{sub}")

###Check

In [0]:
spark.sql("""
  SELECT symbol, trade_date, close, volume, price_change_7d, price_change_30d
  FROM jrvs_databricks_fundamentals.default.daily_gold
  ORDER BY symbol, trade_date
  LIMIT 20
""").show()

+------+----------+------+--------+------------------+----------------+
|symbol|trade_date| close|  volume|   price_change_7d|price_change_30d|
+------+----------+------+--------+------------------+----------------+
|  AAPL|2026-03-24|251.64|45152288|              NULL|            NULL|
|  AAPL|2026-03-25|252.62|28476668|              NULL|            NULL|
|  AAPL|2026-03-26|252.89|41796650|              NULL|            NULL|
|  AAPL|2026-03-27| 248.8|47899998|              NULL|            NULL|
|  AAPL|2026-03-30|246.63|39446213|              NULL|            NULL|
|  AAPL|2026-03-31|253.79|49598091|              NULL|            NULL|
|  AAPL|2026-04-01|255.63|40059432|              NULL|            NULL|
|  AAPL|2026-04-02|255.92|31289369| 4.280000000000001|            NULL|
|  AAPL|2026-04-06|258.86|29329911| 6.240000000000009|            NULL|
|  AAPL|2026-04-07| 253.5|62148008|0.6100000000000136|            NULL|
|  AAPL|2026-04-08| 258.9|41032772|10.099999999999966|          